# 03 - Test filters, including their failure modes
Render the configured PDF at the target DPI, then compare median denoising, non-local means, light sharpening and denoise-then-sharpen. A two-row dark-stroke growth operation is test-only. Thinning, Otsu and median filtering of a lower-resolution render are local cautionary demonstrations, not automatic production steps.

The input is always the same scanned PDF. This self-contained notebook reads the local `.env`, renders every page itself and sends each derivative to the same configured Content Understanding analyzer. Image libraries only preprocess; CU performs extraction. Notebook 01 is not needed. Filters can damage a sign as well as preserve it.

In [ ]:
import hashlib, json
from pathlib import Path
import cv2 as cv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pypdfium2 as pdfium
from PIL import Image, ImageFilter
from dotenv import dotenv_values

OCR = Path('..').resolve()
if not (OCR / '.env').is_file():
    raise FileNotFoundError('Copy demo/ocr/.env.example to demo/ocr/.env first.')
config = dotenv_values(OCR / '.env')
PDF = (OCR / config['OCR_INPUT_FILE']).resolve()
PAGE = int(config.get('OCR_FOCUS_PAGE', 1))
DPI = int(config.get('OCR_TARGET_DPI', 300))
ANALYZER = (config.get('CU_ANALYZER_ID') or '').strip()
API = config.get('CU_API_VERSION') or '2025-11-01'
OUT = OCR / 'output'
(OUT / 'versions').mkdir(parents=True, exist_ok=True)
(OUT / 'screenshots').mkdir(exist_ok=True)
original_hash = hashlib.sha256(PDF.read_bytes()).hexdigest()

## Existing library operations, not new filter algorithms
Rendering and multi-page TIFF encoding adapt the [existing renderer](https://github.com/jomedinagomez/azure-ai-foundry-insurance/blob/f6949572c5aaa6bc77b1ac592a92affc62d98346/demo/sov/preprocess/pdf_render.py). Denoising uses [Pillow MedianFilter](https://pillow.readthedocs.io/en/stable/reference/ImageFilter.html#PIL.ImageFilter.MedianFilter) and the [OpenCV non-local-means example](https://github.com/opencv/opencv/blob/55ff746075718ec2d7cfe09b106c070555f3dbff/doc/py_tutorials/py_photo/py_non_local_means/py_non_local_means.markdown). Sharpening uses [Pillow UnsharpMask](https://github.com/python-pillow/Pillow/blob/98fa02d46db2bc96f04521e5989d08a8b6083c9f/src/PIL/ImageFilter.py). Morphology follows [OpenCV's definitions](https://docs.opencv.org/4.x/d9/d61/tutorial_py_morphological_ops.html).

For black marks on white, a minimum operation (`cv.erode`) grows the dark mark; a maximum operation (`cv.dilate`) thins it. This is the inverse of describing white foreground objects. All output is lossless grayscale TIFF, and all input pages are retained. Parameter choices are experiments, not validated settings.

In [ ]:
def render_pages(dpi):
    pages = []
    with pdfium.PdfDocument(str(PDF)) as document:
        for page in document:
            bitmap = page.render(scale=dpi / 72)
            try:
                pages.append(bitmap.to_pil().convert('L'))
            finally:
                bitmap.close()
    return pages

base = render_pages(DPI)
low = render_pages(150)
median = [page.filter(ImageFilter.MedianFilter(3)) for page in base]
variants = {
    f'rendered_{DPI}dpi': base,
    f'median_{DPI}dpi': median,
    f'nlmeans_{DPI}dpi': [Image.fromarray(cv.fastNlMeansDenoising(np.asarray(page), None, h=3, templateWindowSize=7, searchWindowSize=21)) for page in base],
    f'sharpened_{DPI}dpi': [page.filter(ImageFilter.UnsharpMask(radius=1, percent=100, threshold=2)) for page in base],
    f'denoise_sharpen_{DPI}dpi': [page.filter(ImageFilter.UnsharpMask(radius=1, percent=100, threshold=2)) for page in median],
    f'thickened_{DPI}dpi': [Image.fromarray(cv.erode(np.asarray(page), np.ones((2, 1), np.uint8))) for page in base],
}
cautions = {
    'median_on_150dpi_render': [page.filter(ImageFilter.MedianFilter(3)) for page in low],
    f'thinned_{DPI}dpi': [Image.fromarray(cv.dilate(np.asarray(page), np.ones((2, 1), np.uint8))) for page in base],
    f'otsu_{DPI}dpi': [Image.fromarray(cv.threshold(np.asarray(page), 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)[1]) for page in base],
}
def save_lossless(pages, path, resolution):
    if path.suffix.casefold() not in ('.tif', '.tiff'):
        raise ValueError('These multi-page variants must be saved as lossless TIFF, never JPEG.')
    if path.exists():
        with Image.open(path) as saved:
            identical = saved.n_frames == len(pages)
            if identical:
                for index, page in enumerate(pages):
                    saved.seek(index)
                    if saved.mode != page.mode or saved.size != page.size or saved.tobytes() != page.tobytes() or tuple(saved.info.get('dpi', ())) != (resolution, resolution):
                        identical = False
                        break
            if identical:
                return
    pages[0].save(path, save_all=True, append_images=pages[1:], compression='tiff_lzw', dpi=(resolution, resolution))

inputs = {}
for label, pages in {**variants, **cautions}.items():
    path = OUT / 'versions' / f'{label}.tiff'
    resolution = 150 if label == 'median_on_150dpi_render' else DPI
    save_lossless(pages, path, resolution)
    inputs[label] = path
print('Pages retained per variant:', len(base))

## Capture comparable crops and a pixel diagnostic
All crops use the same physical coordinates from `.env`. The lower-resolution example is a 150 dpi visible-page render, not an extracted raw fax. A dark-pixel count in a manually specified minus box is only a local image diagnostic: surrounding pixels, interpolation and threshold affect it. Do not equate it with OCR accuracy or force it to zero to match a synthetic demonstration.

In [ ]:
box = tuple(float(v) for v in config['OCR_CROP_BOX'].split(','))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for axis, (label, pages) in zip(axes.flat, variants.items()):
    crop = pages[PAGE - 1].crop(tuple(round(v * DPI) for v in box))
    axis.imshow(crop, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    axis.set_title(label)
    axis.axis('off')
fig.tight_layout()
fig.savefig(OUT / 'screenshots' / 'filter_comparison.png', dpi=140)
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for axis, (label, pages) in zip(axes, cautions.items()):
    resolution = 150 if label == 'median_on_150dpi_render' else DPI
    crop = pages[PAGE - 1].crop(tuple(round(v * resolution) for v in box))
    axis.imshow(crop, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    axis.set_title(label + ' - caution')
    axis.axis('off')
fig.tight_layout()
fig.savefig(OUT / 'screenshots' / 'caution_comparison.png', dpi=140)
plt.show()
pixel_rows = []
if config.get('OCR_MINUS_BOX'):
    minus_box = tuple(float(v) for v in config['OCR_MINUS_BOX'].split(','))
    for label, pages in {**variants, **cautions}.items():
        resolution = 150 if label == 'median_on_150dpi_render' else DPI
        crop = pages[PAGE - 1].crop(tuple(round(v * resolution) for v in minus_box))
        dark = np.asarray(crop) < 128
        pixel_rows.append({'case': label, 'dark_pixels': int(dark.sum()), 'box_pixels': int(dark.size), 'dark_fraction': round(float(dark.mean()), 4)})
    pd.DataFrame(pixel_rows).to_csv(OUT / 'minus_pixel_checks.csv', index=False)
    display(pd.DataFrame(pixel_rows))
else:
    print('Minus-box diagnostic not configured; no pixel-count claim is made.')

## Send the six scan variants to the same CU analyzer
Uses the same [official SDK request](https://github.com/Azure/azure-sdk-for-python/blob/53023a2cd9adf255cff520a99985fff84dc58d7d/sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_analyze_binary.py), local `.env`, cache and analyzer as notebooks 01/02. [MIT notice](../LICENSE). The control reuses its response if byte-identical. Cache misses make billable requests. Caution-only variants are saved and displayed, not submitted here.

In [ ]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential

endpoint = config.get('CONTENTUNDERSTANDING_ENDPOINT', '').rstrip('/')
if endpoint and not ANALYZER:
    raise ValueError('Set CU_ANALYZER_ID in demo/ocr/.env to the existing analyzer used for extraction.')
key = config.get('CONTENTUNDERSTANDING_KEY')
client = ContentUnderstandingClient(endpoint=endpoint, credential=AzureKeyCredential(key) if key else DefaultAzureCredential(process_timeout=60), api_version=API) if endpoint else None

def analyze(path, label):
    if client is None:
        print('Skipped: set CONTENTUNDERSTANDING_ENDPOINT in demo/ocr/.env')
        return None
    data = path.read_bytes()
    digest = hashlib.sha256(data + f'{endpoint}|{ANALYZER}|{API}'.encode()).hexdigest()
    cache = OUT / 'cu-cache' / f'{digest}.json'
    cache.parent.mkdir(exist_ok=True)
    if cache.exists():
        result = json.loads(cache.read_text(encoding='utf-8'))
        print(label, '- cached')
    else:
        print(label, '- submitting to', ANALYZER)
        result = client.begin_analyze_binary(analyzer_id=ANALYZER, binary_input=data).result().as_dict()
        cache.write_text(json.dumps(result, indent=2), encoding='utf-8')
    (OUT / f'{label}.md').write_text('\n\n'.join(item.get('markdown', '') for item in result['contents']), encoding='utf-8')
    (OUT / f'{label}.input.json').write_text(json.dumps({'input_sha256': hashlib.sha256(data).hexdigest(), 'cache': cache.name, 'analyzer': ANALYZER, 'api_version': API}, indent=2), encoding='utf-8')
    return result

responses = {label: analyze(inputs[label], label) for label in variants}

In [ ]:
keyword = config.get('OCR_KEYWORD', 'property tax').casefold()
for label, result in responses.items():
    print('\n', label)
    if result is None:
        print('Not analyzed.')
        continue
    lines = '\n'.join(item.get('markdown', '') for item in result['contents']).splitlines()
    matches = [i for i, line in enumerate(lines) if keyword in line.casefold()]
    if not matches:
        print('Label not found; inspect saved Markdown.')
    for i in matches[:2]:
        print('\n'.join(lines[max(0, i - 1):i + 3]))
for pages in [*variants.values(), *cautions.values(), low]:
    for page in pages:
        page.close()
print('Original unchanged:', hashlib.sha256(PDF.read_bytes()).hexdigest() == original_hash)

## Takeaways
Keep a filter only if CU extraction improves without unacceptable regressions. Notebook 04 compares the original scanned PDF and these variants, using the same analyzer and saved responses. It shows returned fields separately from Markdown sign checks. One passing scan is not a production benchmark. If the original already keeps the sign, the result is preservation, not a repair.